1. 환경 설정 및 라이브러리 설치 (Cell 1)

In [ ]:
import sys, transformers, pkgutil, importlib
print("transformers version:", transformers.__version__)
print("transformers path:", transformers.__file__)

# 충돌나는 모듈 확인
spec = pkgutil.find_loader("transformers.generation")
print("generation module:", spec.path if spec else None)


In [ ]:
!pip install transformers datasets sentencepiece accelerate torch

import datasets
from datasets import load_dataset, DatasetDict
from transformers import pipeline
import torch
import pandas as pd # 데이터 확인용

# GPU 사용 가능 여부 확인 및 설정 (Colab에서는 보통 GPU 사용 가능)
device = 0 if torch.cuda.is_available() else -1
print(f"사용 가능한 디바이스: {'GPU' if device == 0 else 'CPU'}")

2. 데이터셋 로드 및 준비 (Cell 2)


In [ ]:
ds = load_dataset("stanfordnlp/imdb")
print(ds)
print(ds["train"][0])

In [ ]:
# 실습용 subset
num_samples_to_use = 200
imdb_subset = ds["train"].select(range(num_samples_to_use))

print("로드된 데이터셋 정보:")
print(imdb_subset)

print("\n첫 번째 데이터 예시 (ctext 확인):")
print(imdb_subset[0]['text'])

# 데이터셋 확인을 위해 Pandas DataFrame으로 변환 (선택 사항)
df_check = pd.DataFrame(imdb_subset)
print("\n데이터셋 일부 미리보기 (DataFrame):")
display(df_check.head(3)) # Colab 환경에서는 display()가 표 형태로 보여줍니다.

3. 감정 분석 모델 파이프라인 로드

In [ ]:
emotion_classifier = pipeline(
    task = "text-classification",
    model = "SamLowe/roberta-base-go_emotions",
    device = device,
    top_k = 1,
    truncation=True,     # 긴 입력 자르기
    padding=True,        # 배치 패딩
    max_length=512       # RoBERTa 최대 길이
)

print("감정 분석 파이프라인 로드 완료.")

# 감정 분석 테스트
test_emotion = emotion_classifier("This has to be the worst piece of garbage.")
print(f"감정 분석 테스트: {test_emotion}")

4. 감정 분석 및 데이터셋에 추가

In [ ]:
def analyze_emotion(example):
    out = emotion_classifier(example["text"])
    # 방어적 처리: top_k 사용 시 list[list[dict]], 미사용 시 list[dict]
    if isinstance(out, list) and len(out) > 0 and isinstance(out[0], list):
        label = out[0][0]["label"]
        score = out[0][0]["score"]
    else:
        label = out[0]["label"]
        score = out[0]["score"]
    return {"emotion": label, "emotion_score": float(score)}

print("감정 분석 작업을 시작합니다...")
final_dataset = imdb_subset.map(analyze_emotion)
print("감정 분석 작업 완료.")

print("\n최종 데이터셋 정보:")
print(final_dataset)

print("\n첫 번째 데이터의 영어 번역(english_summary)과 감정(emotion):")
print("--- 영어 번역 (English Summary) ---")
print(final_dataset[0]['text'])
print("\n--- 분석된 감정 (Emotion) ---")
print(final_dataset[0]['emotion'], f"(score={final_dataset[0]['emotion_score']:.4f})")

# 최종 데이터셋 확인 (Pandas)
df_final = pd.DataFrame(final_dataset)
print("\n최종 데이터셋 미리보기 (DataFrame):")
display(df_final) # 전체 선택된 샘플 표시

5. 결과 정리 및 마무리

In [ ]:
df_text_emotion = df_final[['text', 'emotion']].copy()

print("새로운 DataFrame (text, emotion):")
display(df_text_emotion.head(5))

print("모든 작업이 완료되었습니다.")
# print("최종 데이터셋 컬럼:", final_dataset.column_names)

# 필요시 CSV 저장
# df_final.to_csv("klue_mrc_processed.csv", index=False, encoding='utf-8-sig')
# print("\n결과를 CSV 파일로 저장했습니다. (필요시 주석 해제)")